# **Design Patterns in Python**

---


### Learning Objectives

By the end of this notebook, you will be able to:

1. Understand what design patterns are and why they matter

2. Implement **Creational Patterns**: Singleton, Factory, Builder

3. Implement **Structural Patterns**: Adapter, Decorator, Facade

4. Implement **Behavioral Patterns**: Observer, Strategy, Command

5. Choose the right pattern for real-world problems

6. Recognize patterns in popular Python libraries

---

### 📚 What are Design Patterns?

> "Design patterns are **reusable solutions** to commonly occurring problems in software design."

Think of them as **recipes** — proven approaches that experienced developers have used and refined over time.

#### Why Learn Design Patterns?

| Benefit | Description |
|---------|-------------|
| **Common Vocabulary** | "Use the Observer pattern" is clearer than explaining the entire mechanism |
| **Proven Solutions** | Avoid reinventing the wheel |
| **Readable Code** | Other developers instantly understand your intent |
| **Testable Code** | Patterns often lead to loosely coupled, testable designs |
| **Interview Ready** | Design patterns are common interview topics! |

---

## 🏭 Part 1: Creational Patterns

**Creational patterns** deal with object creation mechanisms — trying to create objects in a manner suitable to the situation.

---

### 1.1 Singleton Pattern 🔒

**Intent:** Ensure a class has only ONE instance, and provide a global point of access to it.

#### Real-World Examples:
- 🖥️ **Database Connection Pool** — Only one pool manages all connections
- ⚙️ **Application Configuration** — One config object for the entire app
- 📝 **Logger** — One logger instance for consistent logging

#### When to Use:
- When exactly one object is needed to coordinate actions across the system
- When you need strict control over global variables

In [1]:
# ❌ Problem: Multiple database connections waste resources

class DatabaseConnection:
    def __init__(self):
        print("🔌 Creating new database connection...")
        self.connected = True

# Every call creates a NEW connection - wasteful!
conn1 = DatabaseConnection()
conn2 = DatabaseConnection()
conn3 = DatabaseConnection()

print(f"\nAre they the same object? {conn1 is conn2}")  # False - different objects!

🔌 Creating new database connection...
🔌 Creating new database connection...
🔌 Creating new database connection...

Are they the same object? False


In [2]:
# ✅ Solution: Singleton Pattern

class DatabaseConnection:
    _instance = None  # Class-level storage for the single instance
    
    def __new__(cls):
        """Called BEFORE __init__. Controls object creation."""
        if cls._instance is None:
            print("🔌 Creating database connection (first time only!)")
            cls._instance = super().__new__(cls)
            cls._instance.connected = True
        return cls._instance
    
    def query(self, sql):
        return f"Executing: {sql}"

# Test it!
conn1 = DatabaseConnection()
conn2 = DatabaseConnection()
conn3 = DatabaseConnection()

print(f"\nAre they the same object? {conn1 is conn2}")  # True!
print(f"Same IDs? {id(conn1)} == {id(conn2)} == {id(conn3)}")

🔌 Creating database connection (first time only!)

Are they the same object? True
Same IDs? 4426910784 == 4426910784 == 4426910784


#### 🐍 Pythonic Singleton using Module

In Python, the **simplest singleton is just a module**! Module-level variables are only initialized once.

In [ ]:
# config.py (conceptual - as a module)
# This IS a singleton - Python modules are only loaded once!

class _Config:
    def __init__(self):
        self.debug = True
        self.database_url = "localhost:5432"
        self.api_key = "secret_key_123"

# The single instance
config = _Config()

# Now anyone can: from config import config
print(f"Debug mode: {config.debug}")
print(f"Database: {config.database_url}")

#### 💡 Try It: Singleton Logger

In [5]:
# TODO: Create a Singleton Logger class
# Requirements:
# - Only ONE logger instance should exist
# - Should have a log(message) method that prints with a timestamp
# - Should track how many messages have been logged

from datetime import datetime

class Logger:
    _instance = None
    
    def __new__(cls):
        if cls._instance is None:
            cls._instance = super().__new__(cls)
            cls._instance.message_count = 0
        return cls._instance
    
    def log(self, message):
        print(f"[{datetime.now()}] {message}")
        self.message_count += 1

# Test
logger1 = Logger()
logger2 = Logger()

logger1.log("First message")
logger2.log("Second message")
logger1.log("Third message")
print(f"Same instance? {logger1 is logger2}")  # Should be True
print(f"Total logs: {logger1.message_count}")  # Should be 2

[2026-03-05 17:44:29.055561] First message
[2026-03-05 17:44:29.055887] Second message
[2026-03-05 17:44:29.056019] Third message
Same instance? True
Total logs: 3


---

### 1.2 Factory Pattern 🏭

**Intent:** Create objects without specifying the exact class to create. Let subclasses or factory methods decide.

#### Real-World Examples:
- 🎮 **Game Characters** — Create warriors, mages, or archers based on user choice
- 📄 **Document Parsers** — Create PDF, Word, or JSON parser based on file type
- 💳 **Payment Processors** — Create Stripe, PayPal, or Square processor

#### When to Use:
- When you don't know ahead of time what class you need
- When you want to delegate creation logic to specialized methods

In [ ]:
# Simple Factory Example: Notification System

from abc import ABC, abstractmethod

# Abstract base class
class Notification(ABC):
    @abstractmethod
    def send(self, message: str) -> str:
        pass

# Concrete implementations
class EmailNotification(Notification):
    def send(self, message: str) -> str:
        return f"📧 Email sent: {message}"

class SMSNotification(Notification):
    def send(self, message: str) -> str:
        return f"📱 SMS sent: {message}"

class PushNotification(Notification):
    def send(self, message: str) -> str:
        return f"🔔 Push notification: {message}"

# The Factory!
class NotificationFactory:
    @staticmethod
    def create(notification_type: str) -> Notification:
        """Factory method - creates the right notification type."""
        factories = {
            "email": EmailNotification,
            "sms": SMSNotification,
            "push": PushNotification
        }
        
        if notification_type not in factories:
            raise ValueError(f"Unknown notification type: {notification_type}")
        
        return factories[notification_type]()

# Usage - the client doesn't need to know concrete classes!
notification = NotificationFactory.create("email")
print(notification.send("Hello, World!"))

notification = NotificationFactory.create("sms")
print(notification.send("Your code is 1234"))

notification = NotificationFactory.create("push")
print(notification.send("New message received!"))

---

### 1.3 Builder Pattern 🏗️

**Intent:** Construct complex objects step by step. Allows you to produce different types and representations using the same construction process.

#### Real-World Examples:
- 🍔 **Meal Builder** — Build a burger with optional toppings
- 🏠 **House Builder** — Construct houses with varying features
- 📊 **Query Builder** — Build SQL queries piece by piece
- 📧 **Email Builder** — Construct emails with optional CC, BCC, attachments

#### When to Use:
- When object creation involves many optional parameters
- When you want to avoid "telescoping constructors" (many __init__ parameters)

In [ ]:
# ❌ Problem: Too many constructor parameters!

class Pizza:
    def __init__(self, size, cheese=True, pepperoni=False, mushrooms=False,
                 onions=False, bacon=False, olives=False, extra_cheese=False,
                 stuffed_crust=False, thin_crust=False):
        self.size = size
        self.cheese = cheese
        self.pepperoni = pepperoni
        self.mushrooms = mushrooms
        self.onions = onions
        self.bacon = bacon
        self.olives = olives
        self.extra_cheese = extra_cheese
        self.stuffed_crust = stuffed_crust
        self.thin_crust = thin_crust

# Hard to read and error-prone!
pizza = Pizza("large", True, True, False, True, True, False, True, False, False)
# What does the 5th True mean? 🤔

In [6]:
# ✅ Solution: Builder Pattern

class Pizza:
    def __init__(self):
        self.size = "medium"
        self.toppings = []
        self.crust = "regular"
    
    def __str__(self):
        toppings_str = ", ".join(self.toppings) if self.toppings else "no toppings"
        return f"🍕 {self.size.title()} pizza with {self.crust} crust and {toppings_str}"


class PizzaBuilder:
    """Builds a pizza step by step with a fluent interface."""
    
    def __init__(self):
        self.pizza = Pizza()
    
    def set_size(self, size: str) -> 'PizzaBuilder':
        self.pizza.size = size
        return self  # Return self for chaining!
    
    def add_pepperoni(self) -> 'PizzaBuilder':
        self.pizza.toppings.append("🔴 pepperoni")
        return self
    
    def add_mushrooms(self) -> 'PizzaBuilder':
        self.pizza.toppings.append("🍄 mushrooms")
        return self
    
    def add_onions(self) -> 'PizzaBuilder':
        self.pizza.toppings.append("🧅 onions")
        return self
    
    def add_bacon(self) -> 'PizzaBuilder':
        self.pizza.toppings.append("🥓 bacon")
        return self
    
    def add_extra_cheese(self) -> 'PizzaBuilder':
        self.pizza.toppings.append("🧀 extra cheese")
        return self
    
    def set_thin_crust(self) -> 'PizzaBuilder':
        self.pizza.crust = "thin"
        return self
    
    def set_stuffed_crust(self) -> 'PizzaBuilder':
        self.pizza.crust = "stuffed"
        return self
    
    def build(self) -> Pizza:
        """Return the final pizza."""
        return self.pizza


# Beautiful, readable pizza creation!
my_pizza = (PizzaBuilder()
    .set_size("large")
    .add_pepperoni()
    .add_mushrooms()
    .add_extra_cheese()
    .set_stuffed_crust()
    .build())

print(my_pizza)

🍕 Large pizza with stuffed crust and 🔴 pepperoni, 🍄 mushrooms, 🧀 extra cheese


---

## 🔗 Part 2: Structural Patterns

**Structural patterns** deal with object composition — how classes and objects can be combined to form larger structures.

---

### 2.1 Adapter Pattern 🔌

**Intent:** Convert the interface of a class into another interface that clients expect. Let incompatible interfaces work together.

#### Real-World Analogy:
- 🔌 **Power Adapter** — US plug → European socket
- 🎧 **Audio Adapter** — 3.5mm jack → USB-C

#### When to Use:
- When you need to use an existing class, but its interface doesn't match what you need
- When integrating third-party libraries

In [ ]:
# 🔌 Adapter Pattern: Integrating a legacy payment system

# Old legacy system (can't modify this!)
class LegacyPaymentSystem:
    """Old payment system with different method names."""
    def make_payment(self, amount_cents: int, card_number: str) -> bool:
        print(f"💳 Legacy: Processing {amount_cents} cents from card {card_number[-4:]}")
        return True

# New interface that our app expects
class PaymentProcessor(ABC):
    @abstractmethod
    def process(self, amount: float, payment_info: dict) -> bool:
        pass

# The Adapter - bridges old and new!
class LegacyPaymentAdapter(PaymentProcessor):
    """Adapts the legacy system to our new interface."""
    
    def __init__(self, legacy_system: LegacyPaymentSystem):
        self.legacy = legacy_system
    
    def process(self, amount: float, payment_info: dict) -> bool:
        # Convert dollars to cents
        amount_cents = int(amount * 100)
        # Extract card number from payment_info
        card_number = payment_info.get("card_number", "")
        # Call the legacy system
        return self.legacy.make_payment(amount_cents, card_number)

# Usage
legacy = LegacyPaymentSystem()
adapter = LegacyPaymentAdapter(legacy)

# Now we can use the new interface!
result = adapter.process(
    amount=99.99,
    payment_info={"card_number": "4111111111111234", "cvv": "123"}
)
print(f"Payment successful: {result}")

---

### 2.2 Decorator Pattern 🎀

**Intent:** Attach additional responsibilities to an object dynamically. Provide a flexible alternative to subclassing.

> ⚠️ **Note:** This is different from Python's `@decorator` syntax, though they share the same concept!

#### Real-World Examples:
- ☕ **Coffee Shop** — Base coffee + milk + sugar + whipped cream
- 🎮 **Game Power-ups** — Character + speed boost + shield + double damage
- 📝 **Text Formatting** — Text + bold + italic + underline

In [ ]:
# 🎀 Decorator Pattern: Coffee Shop

from abc import ABC, abstractmethod

# Base component
class Coffee(ABC):
    @abstractmethod
    def get_description(self) -> str:
        pass
    
    @abstractmethod
    def get_cost(self) -> float:
        pass

# Concrete component
class Espresso(Coffee):
    def get_description(self) -> str:
        return "☕ Espresso"
    
    def get_cost(self) -> float:
        return 2.00

# Base decorator
class CoffeeDecorator(Coffee):
    def __init__(self, coffee: Coffee):
        self._coffee = coffee

# Concrete decorators
class Milk(CoffeeDecorator):
    def get_description(self) -> str:
        return f"{self._coffee.get_description()} + 🥛 Milk"
    
    def get_cost(self) -> float:
        return self._coffee.get_cost() + 0.50

class Sugar(CoffeeDecorator):
    def get_description(self) -> str:
        return f"{self._coffee.get_description()} + 🍬 Sugar"
    
    def get_cost(self) -> float:
        return self._coffee.get_cost() + 0.25

class WhippedCream(CoffeeDecorator):
    def get_description(self) -> str:
        return f"{self._coffee.get_description()} + 🍨 Whipped Cream"
    
    def get_cost(self) -> float:
        return self._coffee.get_cost() + 0.75

# Build a fancy coffee!
my_coffee = Espresso()  # Start with base
my_coffee = Milk(my_coffee)  # Add milk
my_coffee = Sugar(my_coffee)  # Add sugar
my_coffee = WhippedCream(my_coffee)  # Add whipped cream

print(my_coffee.get_description())
print(f"Total: ${my_coffee.get_cost():.2f}")

---

### 2.3 Facade Pattern 🏛️

**Intent:** Provide a unified, simplified interface to a set of complex subsystems.

#### Real-World Examples:
- 🎬 **Home Theater** — One "watch movie" button controls TV, speakers, lights, projector
- 🏦 **Bank Withdrawal** — Simple interface hides account verification, balance check, ledger update
- 🖥️ **Computer Startup** — One button powers on CPU, RAM, hard drive, etc.

In [ ]:
# 🏛️ Facade Pattern: Home Theater System

# Complex subsystems
class TV:
    def on(self): print("📺 TV is ON")
    def off(self): print("📺 TV is OFF")
    def set_input(self, source): print(f"📺 Input set to {source}")

class SoundSystem:
    def on(self): print("🔊 Sound system is ON")
    def off(self): print("🔊 Sound system is OFF")
    def set_volume(self, level): print(f"🔊 Volume set to {level}")
    def set_surround_mode(self): print("🔊 Surround sound activated")

class StreamingPlayer:
    def on(self): print("▶️ Streaming player is ON")
    def off(self): print("▶️ Streaming player is OFF")
    def play(self, movie): print(f"▶️ Playing: {movie}")
    def pause(self): print("▶️ Paused")

class Lights:
    def dim(self, level): print(f"💡 Lights dimmed to {level}%")
    def on(self): print("💡 Lights ON at 100%")

# The Facade - simple interface!
class HomeTheaterFacade:
    def __init__(self):
        self.tv = TV()
        self.sound = SoundSystem()
        self.player = StreamingPlayer()
        self.lights = Lights()
    
    def watch_movie(self, movie: str):
        """One simple method to start movie night!"""
        print("\n🎬 Starting Movie Night...\n")
        self.lights.dim(10)
        self.tv.on()
        self.tv.set_input("HDMI1")
        self.sound.on()
        self.sound.set_surround_mode()
        self.sound.set_volume(65)
        self.player.on()
        self.player.play(movie)
        print("\n🍿 Enjoy your movie!\n")
    
    def end_movie(self):
        """Shut everything down."""
        print("\n🛑 Ending Movie Night...\n")
        self.player.off()
        self.sound.off()
        self.tv.off()
        self.lights.on()

# Simple usage!
theater = HomeTheaterFacade()
theater.watch_movie("The Matrix")

---

## 🎭 Part 3: Behavioral Patterns

**Behavioral patterns** deal with communication between objects — how objects interact and distribute responsibility.

---

### 3.1 Observer Pattern 👀

**Intent:** Define a one-to-many dependency between objects so that when one object changes state, all its dependents are notified automatically.

#### Also Known As:
- Publish-Subscribe (Pub/Sub)
- Event System
- Listener Pattern

#### Real-World Examples:
- 📰 **Newsletter** — Subscribers get notified of new articles
- 📊 **Stock Prices** — Multiple displays update when price changes
- 🔔 **Social Media** — Followers notified of new posts

In [ ]:
# 👀 Observer Pattern: Weather Station

from abc import ABC, abstractmethod
from typing import List

# Observer interface
class WeatherObserver(ABC):
    @abstractmethod
    def update(self, temperature: float, humidity: float):
        pass

# Subject (Observable)
class WeatherStation:
    def __init__(self):
        self._observers: List[WeatherObserver] = []
        self._temperature = 0.0
        self._humidity = 0.0
    
    def subscribe(self, observer: WeatherObserver):
        self._observers.append(observer)
        print(f"✅ {observer.__class__.__name__} subscribed")
    
    def unsubscribe(self, observer: WeatherObserver):
        self._observers.remove(observer)
        print(f"❌ {observer.__class__.__name__} unsubscribed")
    
    def notify_observers(self):
        for observer in self._observers:
            observer.update(self._temperature, self._humidity)
    
    def set_measurements(self, temperature: float, humidity: float):
        print(f"\n🌡️ Weather Station: New data - {temperature}°C, {humidity}% humidity")
        self._temperature = temperature
        self._humidity = humidity
        self.notify_observers()  # Automatically notify all observers!

# Concrete Observers
class PhoneDisplay(WeatherObserver):
    def update(self, temperature: float, humidity: float):
        print(f"   📱 Phone: It's {temperature}°C outside")

class WindowDisplay(WeatherObserver):
    def update(self, temperature: float, humidity: float):
        print(f"   🖼️ Window Display: {temperature}°C | {humidity}% humidity")

class AlertSystem(WeatherObserver):
    def update(self, temperature: float, humidity: float):
        if temperature > 35:
            print(f"   🚨 ALERT: Extreme heat warning! {temperature}°C")
        elif temperature < 0:
            print(f"   🚨 ALERT: Freezing warning! {temperature}°C")
        else:
            print(f"   ✅ Alert System: Temperature normal")

# Demo
station = WeatherStation()

# Create and subscribe observers
phone = PhoneDisplay()
window = WindowDisplay()
alert = AlertSystem()

station.subscribe(phone)
station.subscribe(window)
station.subscribe(alert)

# Update weather - all observers get notified!
station.set_measurements(22.5, 65)
station.set_measurements(38.0, 40)

---

### 3.2 Strategy Pattern 🎯

**Intent:** Define a family of algorithms, encapsulate each one, and make them interchangeable. Strategy lets the algorithm vary independently from clients that use it.

#### Real-World Examples:
- 💳 **Payment Methods** — Pay by card, cash, or mobile
- 🚗 **Navigation** — Route by fastest, shortest, or scenic
- 📦 **Shipping** — Choose express, standard, or economy
- 🔐 **Authentication** — Login via password, OAuth, or 2FA

In [ ]:
# 🎯 Strategy Pattern: Shipping Cost Calculator

from abc import ABC, abstractmethod
from dataclasses import dataclass

# Strategy interface
class ShippingStrategy(ABC):
    @abstractmethod
    def calculate(self, weight: float) -> float:
        """Calculate shipping cost based on weight in kg."""
        pass
    
    @property
    @abstractmethod
    def name(self) -> str:
        pass

# Concrete strategies
class StandardShipping(ShippingStrategy):
    def calculate(self, weight: float) -> float:
        return weight * 2.50  # $2.50 per kg
    
    @property
    def name(self) -> str:
        return "📦 Standard (5-7 days)"

class ExpressShipping(ShippingStrategy):
    def calculate(self, weight: float) -> float:
        return weight * 5.00 + 10.00  # $5 per kg + $10 handling
    
    @property
    def name(self) -> str:
        return "🚀 Express (1-2 days)"

class PickupStrategy(ShippingStrategy):
    def calculate(self, weight: float) -> float:
        return 0.00  # Free!
    
    @property
    def name(self) -> str:
        return "🏪 Store Pickup (Free)"

# Context - uses a strategy
@dataclass
class Order:
    items: list
    weight: float
    shipping_strategy: ShippingStrategy = None
    
    def set_shipping(self, strategy: ShippingStrategy):
        self.shipping_strategy = strategy
    
    def get_shipping_cost(self) -> float:
        if not self.shipping_strategy:
            raise ValueError("Please select a shipping method!")
        return self.shipping_strategy.calculate(self.weight)
    
    def __str__(self):
        shipping_cost = self.get_shipping_cost()
        return (f"Order: {self.items}\n"
                f"Weight: {self.weight}kg\n"
                f"Method: {self.shipping_strategy.name}\n"
                f"Shipping: ${shipping_cost:.2f}")

# Usage
order = Order(items=["Laptop", "Mouse"], weight=3.5)

# Try different strategies
print("=" * 40)
order.set_shipping(StandardShipping())
print(order)

print("\n" + "=" * 40)
order.set_shipping(ExpressShipping())
print(order)

print("\n" + "=" * 40)
order.set_shipping(PickupStrategy())
print(order)

---

### 3.3 Command Pattern 📜

**Intent:** Encapsulate a request as an object, letting you parameterize, queue, log, and undo operations.

#### Real-World Examples:
- ↩️ **Undo/Redo** — Text editors, image editors
- 🎮 **Game Actions** — Record and replay player moves
- 📋 **Task Queues** — Schedule operations for later execution
- 🔘 **GUI Buttons** — Each button encapsulates an action

In [ ]:
# 📜 Command Pattern: Text Editor with Undo

from abc import ABC, abstractmethod
from typing import List

# Command interface
class Command(ABC):
    @abstractmethod
    def execute(self) -> None:
        pass
    
    @abstractmethod
    def undo(self) -> None:
        pass

# Receiver - the actual text document
class TextDocument:
    def __init__(self):
        self.content = ""
    
    def write(self, text: str):
        self.content += text
    
    def erase(self, num_chars: int):
        self.content = self.content[:-num_chars] if num_chars <= len(self.content) else ""
    
    def __str__(self):
        return f"📄 Document: '{self.content}'"

# Concrete Commands
class WriteCommand(Command):
    def __init__(self, document: TextDocument, text: str):
        self.document = document
        self.text = text
    
    def execute(self):
        self.document.write(self.text)
        print(f"✍️ Wrote: '{self.text}'")
    
    def undo(self):
        self.document.erase(len(self.text))
        print(f"↩️ Undid write of: '{self.text}'")

# Invoker - manages command execution and history
class TextEditor:
    def __init__(self, document: TextDocument):
        self.document = document
        self.history: List[Command] = []
    
    def execute(self, command: Command):
        command.execute()
        self.history.append(command)
    
    def undo(self):
        if self.history:
            command = self.history.pop()
            command.undo()
        else:
            print("⚠️ Nothing to undo!")

# Demo
doc = TextDocument()
editor = TextEditor(doc)

editor.execute(WriteCommand(doc, "Hello "))
print(doc)

editor.execute(WriteCommand(doc, "World!"))
print(doc)

editor.execute(WriteCommand(doc, " How are you?"))
print(doc)

print("\n--- Undo operations ---")
editor.undo()
print(doc)

editor.undo()
print(doc)

---

## 🎯 Design Patterns in the Wild: Python Libraries

You're already using design patterns without knowing it!

| Pattern | Where You've Seen It |
|---------|---------------------|
| **Singleton** | `logging.getLogger()`, `None`, `True`, `False` |
| **Factory** | `datetime.strptime()`, `json.loads()` |
| **Builder** | `pandas.DataFrame().query().groupby().agg()` |
| **Iterator** | `for x in collection:` |
| **Decorator** | `@property`, `@staticmethod`, `@functools.lru_cache` |
| **Observer** | Django signals, GUI event handlers |
| **Strategy** | `sorted(key=...)`, `list.sort(key=...)` |

In [ ]:
# Strategy pattern in action - sorted() with different strategies!

students = [
    {"name": "Alice", "gpa": 3.8, "age": 20},
    {"name": "Bob", "gpa": 3.2, "age": 22},
    {"name": "Charlie", "gpa": 3.9, "age": 19},
]

# Strategy 1: Sort by name
by_name = sorted(students, key=lambda s: s["name"])
print("By name:", [s["name"] for s in by_name])

# Strategy 2: Sort by GPA (descending)
by_gpa = sorted(students, key=lambda s: s["gpa"], reverse=True)
print("By GPA:", [f"{s['name']}: {s['gpa']}" for s in by_gpa])

# Strategy 3: Sort by age
by_age = sorted(students, key=lambda s: s["age"])
print("By age:", [f"{s['name']}: {s['age']}" for s in by_age])

---

## 🏗️ Mini-Project: Task Management System

Use **multiple design patterns** to build a simple task management system!

**Requirements:**
1. **Singleton** - TaskManager (only one instance)
2. **Factory** - Create different task types (Bug, Feature, Improvement)
3. **Observer** - Notify team members when task status changes
4. **Strategy** - Different priority calculation strategies

In [ ]:
# 🏗️ Mini-Project Solution

from abc import ABC, abstractmethod
from datetime import datetime
from typing import List
from dataclasses import dataclass, field

# ============ Observer Pattern ============
class TaskObserver(ABC):
    @abstractmethod
    def on_task_update(self, task: 'Task', old_status: str, new_status: str):
        pass

class TeamMember(TaskObserver):
    def __init__(self, name: str):
        self.name = name
    
    def on_task_update(self, task: 'Task', old_status: str, new_status: str):
        print(f"   📧 {self.name} notified: '{task.title}' changed from {old_status} → {new_status}")

# ============ Strategy Pattern ============
class PriorityStrategy(ABC):
    @abstractmethod
    def calculate(self, task: 'Task') -> int:
        pass

class AgePriorityStrategy(PriorityStrategy):
    """Older tasks get higher priority."""
    def calculate(self, task: 'Task') -> int:
        age_days = (datetime.now() - task.created).days
        return min(age_days + task.base_priority, 10)

class TypePriorityStrategy(PriorityStrategy):
    """Bugs are always higher priority."""
    def calculate(self, task: 'Task') -> int:
        if task.task_type == "Bug":
            return min(task.base_priority + 3, 10)
        return task.base_priority

# ============ Factory Pattern ============
@dataclass
class Task:
    title: str
    task_type: str
    base_priority: int = 5
    status: str = "Open"
    created: datetime = field(default_factory=datetime.now)
    observers: List[TaskObserver] = field(default_factory=list)
    
    def add_observer(self, observer: TaskObserver):
        self.observers.append(observer)
    
    def set_status(self, new_status: str):
        old_status = self.status
        self.status = new_status
        for observer in self.observers:
            observer.on_task_update(self, old_status, new_status)

class TaskFactory:
    @staticmethod
    def create(task_type: str, title: str) -> Task:
        priorities = {"Bug": 8, "Feature": 5, "Improvement": 3}
        return Task(
            title=title,
            task_type=task_type,
            base_priority=priorities.get(task_type, 5)
        )

# ============ Singleton Pattern ============
class TaskManager:
    _instance = None
    
    def __new__(cls):
        if cls._instance is None:
            cls._instance = super().__new__(cls)
            cls._instance.tasks = []
            cls._instance.priority_strategy = TypePriorityStrategy()
        return cls._instance
    
    def add_task(self, task: Task):
        self.tasks.append(task)
        print(f"✅ Added: [{task.task_type}] {task.title}")
    
    def set_priority_strategy(self, strategy: PriorityStrategy):
        self.priority_strategy = strategy
    
    def get_sorted_tasks(self) -> List[Task]:
        return sorted(
            self.tasks,
            key=lambda t: self.priority_strategy.calculate(t),
            reverse=True
        )

# ============ Demo ============
print("=" * 50)
print("🏗️ TASK MANAGEMENT SYSTEM")
print("=" * 50)

# Singleton - get the manager
manager = TaskManager()

# Create team members (observers)
alice = TeamMember("Alice")
bob = TeamMember("Bob")

# Factory - create tasks
bug = TaskFactory.create("Bug", "Login button not working")
feature = TaskFactory.create("Feature", "Add dark mode")
improvement = TaskFactory.create("Improvement", "Optimize database queries")

# Add observers
bug.add_observer(alice)
bug.add_observer(bob)
feature.add_observer(alice)

# Add tasks to manager
manager.add_task(bug)
manager.add_task(feature)
manager.add_task(improvement)

# Observer - update task status
print("\n📢 Status Updates:")
bug.set_status("In Progress")
bug.set_status("Resolved")

# Strategy - see task priorities
print("\n📊 Tasks by Priority (Type Strategy):")
for task in manager.get_sorted_tasks():
    priority = manager.priority_strategy.calculate(task)
    print(f"   [{priority}] [{task.task_type}] {task.title}")

---

## 📝 Practice Exercises

### Exercise 1: Singleton Cache
Create a `Cache` singleton class that stores key-value pairs with an optional TTL (time-to-live).

### Exercise 2: Shape Factory
Create a factory that produces different shapes (Circle, Rectangle, Triangle) with `area()` and `perimeter()` methods.

### Exercise 3: Recipe Builder
Create a `RecipeBuilder` that lets you build recipes step by step:
```python
recipe = (RecipeBuilder()
    .set_name("Pancakes")
    .add_ingredient("Flour", "2 cups")
    .add_ingredient("Milk", "1 cup")
    .add_step("Mix dry ingredients")
    .set_cooking_time(15)
    .build())
```

### Exercise 4: Event System (Observer)
Create an `EventEmitter` class that supports:
- `on(event_name, callback)` - Subscribe to an event
- `emit(event_name, *args)` - Trigger an event
- `off(event_name, callback)` - Unsubscribe

### Exercise 5: Sorting Strategies
Implement different sorting strategies for a `StudentList` class:
- `ByName` - Alphabetically
- `ByGPA` - Highest GPA first
- `ByCredits` - Most credits first

In [ ]:
# Exercise 1: Singleton Cache
# Your code here

class Cache:
    _instance = None
    
    def __new__(cls):
        pass  # Implement singleton
    
    def set(self, key, value, ttl=None):
        pass  # Store value with optional TTL
    
    def get(self, key):
        pass  # Get value (return None if expired)

# Test
# cache1 = Cache()
# cache2 = Cache()
# cache1.set("user", "Alice")
# print(cache2.get("user"))  # Should print "Alice"
# print(cache1 is cache2)  # Should be True

In [ ]:
# Exercise 2: Shape Factory
# Your code here

from abc import ABC, abstractmethod
import math

class Shape(ABC):
    @abstractmethod
    def area(self) -> float:
        pass
    
    @abstractmethod
    def perimeter(self) -> float:
        pass

class Circle(Shape):
    pass  # Implement

class Rectangle(Shape):
    pass  # Implement

class ShapeFactory:
    @staticmethod
    def create(shape_type: str, **kwargs) -> Shape:
        pass  # Implement

# Test
# circle = ShapeFactory.create("circle", radius=5)
# print(f"Circle area: {circle.area():.2f}")
# rect = ShapeFactory.create("rectangle", width=4, height=6)
# print(f"Rectangle perimeter: {rect.perimeter()}")

---

## 📚 Homework

### Problem 1: Plugin System (Factory + Strategy)
Design a plugin system where:
- A `PluginFactory` creates plugins by name
- Each plugin implements a `run()` method
- The main app can load and execute plugins dynamically

### Problem 2: Undo/Redo File Editor (Command)
Extend the command pattern example to support:
- `REDO` functionality
- Multiple command types: `DeleteCommand`, `ReplaceCommand`
- Save command history to a log file

### Problem 3: Stock Market Simulator (Observer + Strategy)
Create a stock market simulator where:
- `Stock` objects notify investors when price changes
- Different trading strategies: `BuyLowSellHigh`, `DayTrader`, `LongTermHolder`
- Each strategy decides whether to buy/sell based on price movements

### Problem 4: Report Generator (Builder)
Create a report builder that generates different formats:
```python
report = (ReportBuilder()
    .set_title("Q4 Sales Report")
    .add_section("Summary", summary_data)
    .add_chart("sales_chart.png")
    .add_table(sales_table)
    .set_footer("Confidential")
    .build(format="pdf"))  # or "html", "markdown"
```

### Problem 5: Design Pattern Analysis (Written)
Choose a Python library you use regularly (e.g., requests, pandas, Django, Flask) and identify **at least 3 design patterns** used in its API. Explain:
- Which pattern is used
- Where in the API you see it
- Why it's a good choice for that situation

---

## 🎉 Summary

| Category | Pattern | Purpose |
|----------|---------|----------|
| **Creational** | Singleton | One instance only |
| | Factory | Create objects without specifying exact class |
| | Builder | Construct complex objects step by step |
| **Structural** | Adapter | Bridge incompatible interfaces |
| | Decorator | Add behavior dynamically |
| | Facade | Simplify complex subsystems |
| **Behavioral** | Observer | Notify dependents of changes |
| | Strategy | Interchangeable algorithms |
| | Command | Encapsulate actions for undo/redo |

### 🔮 Coming Up Next:
- **Decorators & Metaprogramming** — Python's powerful `@decorator` syntax
- **Context Managers** — The `with` statement and `__enter__`/`__exit__`
- **More Patterns** — State, Template Method, Chain of Responsibility